# Análise de notícias em PT-BR com spaCy

Fastcamp de LLM — Card 1, trabalho prático.

**Corpus:** Fake.br-Corpus — 7.200 notícias em português brasileiro, pareadas em
3.600 verdadeiras e 3.600 falsas.

**Ferramenta:** spaCy (`pt_core_news_md`), pipeline completo.

**Pergunta:** o que separa, nos números, uma notícia verdadeira de uma falsa?
Vocabulário, classes gramaticais e entidades nomeadas.

## Bloco 1 — Setup

Rode esta célula uma única vez e **reinicie o runtime** depois (Runtime > Restart session).

In [ ]:
!git clone https://github.com/roneysco/Fake.br-Corpus.git
!python -m spacy download pt_core_news_md

## Bloco 2 — Imports e carga do modelo

Depois do restart, comece por aqui.

In [ ]:
import time
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import spacy
from spacy import displacy

nlp = spacy.load("pt_core_news_md")

print("Componentes do pipeline:", nlp.pipe_names)

## Bloco 3 — Carga do corpus

Os textos estão em `full_texts/true/` e `full_texts/fake/`, um `.txt` por notícia.
Monto um DataFrame com o texto e o rótulo lado a lado.

In [ ]:
BASE = Path("Fake.br-Corpus/full_texts")


def carregar(classe):
    """Lê todos os .txt de uma das pastas e devolve uma lista de dicionários."""
    pasta = BASE / classe
    registros = []
    for arquivo in sorted(pasta.glob("*.txt")):
        texto = arquivo.read_text(encoding="utf-8", errors="ignore").strip()
        if texto:
            registros.append({"arquivo": arquivo.name, "classe": classe, "texto": texto})
    return registros


df = pd.DataFrame(carregar("true") + carregar("fake"))

print("Total de notícias:", len(df))
print(df["classe"].value_counts())
print("Tamanho em caracteres — mediana:", int(df["texto"].str.len().median()))
df.head()

## Bloco 4 — Amostra

Desenvolvo com 200 por classe, que roda em segundos. No final, subo para 3600
e rodo o notebook inteiro uma vez. A `seed` fixa garante que os números não
mudam a cada execução.

In [ ]:
N_POR_CLASSE = 200
SEED = 42

amostra = pd.concat([
    df[df["classe"] == classe].sample(
        min(N_POR_CLASSE, (df["classe"] == classe).sum()),
        random_state=SEED,
    )
    for classe in ["true", "fake"]
]).reset_index(drop=True)

print(amostra["classe"].value_counts())

## Bloco 5 — Processamento

Este é o único passo caro: rodo uma vez e reaproveito os `Doc` em todas as
análises seguintes. `nlp.pipe` processa em lote, que é muito mais rápido que
chamar `nlp(texto)` num laço.

In [ ]:
textos = amostra["texto"].tolist()
classes = amostra["classe"].tolist()

maior = max(len(t) for t in textos)
print(f"Maior texto: {maior} caracteres (limite do spaCy: {nlp.max_length})")

inicio = time.perf_counter()
docs = list(nlp.pipe(textos, batch_size=32))
print(f"{len(docs)} documentos processados em {time.perf_counter() - inicio:.1f}s")

docs_true = [d for d, c in zip(docs, classes) if c == "true"]
docs_fake = [d for d, c in zip(docs, classes) if c == "fake"]

## Bloco 6 — Frequência de lemas

Conto `token.lemma_` em vez da palavra crua: assim "ações" e "ação" contam
junto. Normalizo para minúscula, senão "Brasil" e "brasil" viram entradas
separadas.

In [ ]:
def contar_lemas(lista_docs):
    contador = Counter()
    for doc in lista_docs:
        for token in doc:
            if token.is_stop or token.is_punct or token.is_space:
                continue
            if not token.is_alpha:
                continue
            contador[token.lemma_.lower()] += 1
    return contador


freq_true = contar_lemas(docs_true)
freq_fake = contar_lemas(docs_fake)


def formatar(contador, n=20):
    itens = [f"{palavra} ({qtd})" for palavra, qtd in contador.most_common(n)]
    return itens + [""] * (n - len(itens))


pd.DataFrame({
    "verdadeiras": formatar(freq_true),
    "falsas": formatar(freq_fake),
})

### Comparação crua vs. lematizada

Vale mostrar o efeito da lematização no ranking — é uma das coisas que o
spaCy faz e que uma contagem ingênua de palavras não faz.

In [ ]:
contador_cru = Counter()
for doc in docs_true:
    for token in doc:
        if token.is_stop or token.is_punct or token.is_space or not token.is_alpha:
            continue
        contador_cru[token.text.lower()] += 1

pd.DataFrame({
    "sem lematizar": formatar(contador_cru, 15),
    "lematizado": formatar(freq_true, 15),
})

## Bloco 7 — Classes gramaticais (POS)

Uso **proporção**, não contagem: os dois grupos têm quantidades diferentes de
tokens, então contagem bruta não é comparável.

In [ ]:
def proporcao_pos(lista_docs):
    contador = Counter()
    total = 0
    for doc in lista_docs:
        for token in doc:
            if token.is_punct or token.is_space:
                continue
            contador[token.pos_] += 1
            total += 1
    return {tag: qtd / total for tag, qtd in contador.items()}


pos_df = pd.DataFrame({
    "verdadeiras": proporcao_pos(docs_true),
    "falsas": proporcao_pos(docs_fake),
}).fillna(0)

pos_df["diferenca"] = pos_df["falsas"] - pos_df["verdadeiras"]
pos_df = pos_df.sort_values("verdadeiras", ascending=False)
pos_df.round(4)

In [ ]:
eixo = pos_df[["verdadeiras", "falsas"]].head(10).plot(kind="bar", figsize=(10, 4))
eixo.set_title("Proporção de classes gramaticais por tipo de notícia")
eixo.set_ylabel("proporção de tokens")
eixo.set_xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Bloco 8 — Entidades nomeadas (NER)

O modelo português reconhece `PER` (pessoa), `ORG` (organização), `LOC`
(local) e `MISC` (outros).

In [ ]:
dist_true = Counter(ent.label_ for doc in docs_true for ent in doc.ents)
dist_fake = Counter(ent.label_ for doc in docs_fake for ent in doc.ents)

pd.DataFrame({
    "verdadeiras": dist_true,
    "falsas": dist_fake,
}).fillna(0).astype(int)

In [ ]:
def contar_entidades(lista_docs, rotulo):
    contador = Counter()
    for doc in lista_docs:
        for ent in doc.ents:
            if ent.label_ == rotulo:
                contador[ent.text.strip()] += 1
    return contador


for rotulo in ["PER", "ORG", "LOC"]:
    print(f"\n===== {rotulo} =====")
    tabela = pd.DataFrame({
        "verdadeiras": formatar(contar_entidades(docs_true, rotulo), 10),
        "falsas": formatar(contar_entidades(docs_fake, rotulo), 10),
    })
    print(tabela.to_string(index=False))

## Bloco 9 — Visualização

Um exemplo de cada tipo, com as entidades destacadas.

In [ ]:
exemplo = docs_fake[0]
print("--- trecho de notícia FALSA ---")
displacy.render(exemplo[:80].as_doc(), style="ent", jupyter=True)

In [ ]:
exemplo_true = docs_true[0]
print("--- trecho de notícia VERDADEIRA ---")
displacy.render(exemplo_true[:80].as_doc(), style="ent", jupyter=True)

In [ ]:
# Árvore de dependências de uma sentença (precisa do parser ativo)
sentenca = list(docs_true[0].sents)[0]
displacy.render(
    sentenca.as_doc(),
    style="dep",
    jupyter=True,
    options={"compact": True, "distance": 90},
)

## Bloco 10 — Conclusão

Preencha depois de rodar com a amostra completa:

- **Vocabulário:** quais lemas aparecem em um grupo e não no outro?
- **POS:** qual classe gramatical mais se desloca entre os grupos? Adjetivo e
  advérbio costumam indicar carga opinativa.
- **NER:** os dois grupos falam das mesmas pessoas e organizações? Notícia
  falsa cita menos organizações identificáveis?
- **Limites:** o corpus é de um recorte temporal específico, o modelo foi
  treinado em português moderno de notícia, e amostra de 200 por classe tem
  margem de erro relevante.